# Numerical integration

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. explain how a definite integral can be approximated as a sum of small areas
2. implement left, right and midpoint approximations
3. explain and implement the trapezoidal method
4. compare numerical methods using error and convergence
5. integrate functions and experimental data with SciPy
6. interpret integrals in a chemical context
```

## Integration in chemistry

A definite integral sums contributions over an interval. This makes integration relevant in many chemical situations:

- the area under a chromatographic peak
- the area under an NMR signal
- total heat from a heat-flow signal that changes with time
- integration of rate laws and other differential equations

Numerical integration is especially useful when we have only **discrete measurement data** and no function that we can integrate analytically.

## The rectangle method

The definite integral

$$\int_a^b f(x)\,dx$$

can be approximated by dividing the interval into $n$ small intervals of width

$$h=\frac{b-a}{n}.$$

For each interval we use a rectangle. Its height can be determined from the left endpoint, the right endpoint or the midpoint.

Left approximation:

$$\int_a^b f(x)\,dx\approx h\sum_{k=0}^{n-1}f(a+kh).$$

Right approximation:

$$\int_a^b f(x)\,dx\approx h\sum_{k=1}^{n}f(a+kh).$$

Midpoint approximation:

$$\int_a^b f(x)\,dx\approx h\sum_{k=0}^{n-1}f\left(a+\left(k+\frac12\right)h\right).$$

In [ ]:
def rectangle_left(f, a, b, n):
    h = (b - a) / n
    area = 0.0
    x = a

    for k in range(n):
        area = area + f(x) * h
        x = x + h

    return area


def rectangle_right(f, a, b, n):
    h = (b - a) / n
    area = 0.0
    x = a + h

    for k in range(n):
        area = area + f(x) * h
        x = x + h

    return area


def rectangle_midpoint(f, a, b, n):
    h = (b - a) / n
    area = 0.0
    x = a + h/2

    for k in range(n):
        area = area + f(x) * h
        x = x + h

    return area

We test the methods on

$$f(x)=x^3$$

over the interval $[0,5]$. Here we know the analytical value and can use it as a check:

$$\int_0^5x^3\,dx=\left[\frac{x^4}{4}\right]_0^5=156.25.$$

When we compare a numerical and an analytical result, we must make sure that both calculations use the same integration limits.

In [ ]:
def f(x):
    return x**3

a = 0
b = 5
n = 100
exact = (b**4 - a**4) / 4

print("Left:", rectangle_left(f, a, b, n))
print("Right:", rectangle_right(f, a, b, n))
print("Midpoint:", rectangle_midpoint(f, a, b, n))
print("Exact:", exact)

### Try it yourself

Complete the rectangle methods and investigate how the result changes when you increase the number of rectangles.

<iframe src="../../basthon/?from=examples/numerical_integration_rectangles.py" width="100%" height="620" frameborder="0" title="Try it yourself: numerical integration" loading="lazy" allowfullscreen></iframe>

## Convergence

A numerical answer should not be assessed from a single choice of $n$. We can increase the number of intervals and investigate whether the result **stabilises**.

In [ ]:
n_values = [10, 20, 50, 100, 200, 500, 1000]

print(" n        left error       midpoint error")
for n in n_values:
    error_left = abs(rectangle_left(f, a, b, n) - exact)
    error_mid = abs(rectangle_midpoint(f, a, b, n) - exact)
    print(f"{n:4d}    {error_left:12.6f}    {error_mid:14.6f}")

## The trapezoidal method

Instead of using a flat rectangle top, the trapezoidal method draws a straight line between the function values at the endpoints.

For one interval $[x_i,x_{i+1}]$, the area is

$$A_i\approx\frac{f(x_i)+f(x_{i+1})}{2}h.$$

With $n$ equally wide intervals,

$$\int_a^b f(x)\,dx\approx h\left[\frac{f(a)+f(b)}{2}+\sum_{i=1}^{n-1}f(a+ih)\right].$$

In [ ]:
def trapezoidal_method(f, a, b, n):
    h = (b - a) / n
    area = 0.0
    x = a

    for k in range(n):
        area = area + (f(x) + f(x + h))/2 * h
        x = x + h

    return area

print("Trapezoidal:", trapezoidal_method(f, 0, 5, 100))
print("Exact:", 156.25)

## Simpson's method

Simpson's method uses quadratic polynomials over pairs of intervals. The method requires an **even** number of subintervals and is often very accurate for smooth functions.

$$\int_a^b f(x)\,dx\approx\frac{h}{3}\left[f(a)+f(b)+4\sum_{\text{odd }k}f(x_k)+2\sum_{\text{even }k}f(x_k)\right].$$

In [ ]:
def simpsons_method(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("n must be even.")

    h = (b - a) / n
    total = f(a) + f(b)
    x = a + h

    for k in range(1, n):
        if k % 2 == 0:
            total = total + 2*f(x)
        else:
            total = total + 4*f(x)
        x = x + h

    return total * h / 3

print("Simpson:", simpsons_method(f, 0, 5, 100))

## Using numerical libraries

Once we understand the principle, we can use ready-made functions. In modern SciPy, the relevant functions include:

- `integrate.trapezoid(y, x)` for discrete data
- `integrate.simpson(y, x=x)` for discrete data
- `integrate.quad(f, a, b)` for a function

`trapezoid` and `simpson` are the current names; older code may contain the deprecated names `trapz` and `simps`.

In [ ]:
from scipy import integrate
import numpy as np

x = np.linspace(0, 5, 1001)
y = f(x)

trapezoidal = integrate.trapezoid(y, x)
simpson = integrate.simpson(y, x=x)
quad_value, quad_error = integrate.quad(f, 0, 5)

print("trapezoid:", trapezoidal)
print("simpson:", simpson)
print("quad:", quad_value)
print("estimated absolute error from quad:", quad_error)

## Chemical example: area under a chromatogram

A chromatogram consists of signal as a function of retention time. The area under a peak can be proportional to the amount of substance or concentration after an appropriate calibration.

Here we create a simple synthetic chromatogram with two peaks and integrate the signal numerically. The point is that we are now integrating **measurement points**, not a symbolic function.

In [ ]:
import matplotlib.pyplot as plt

time = np.linspace(0, 10, 501)

peak_1 = 1.2*np.exp(-0.5*((time - 3.0)/0.35)**2)
peak_2 = 0.8*np.exp(-0.5*((time - 6.5)/0.50)**2)
signal = peak_1 + peak_2

plt.plot(time, signal)
plt.xlabel("Retention time (min)")
plt.ylabel("Signal (a.u.)")
plt.show()

In [ ]:
mask_1 = (time >= 2.0) & (time <= 4.2)
mask_2 = (time >= 5.0) & (time <= 8.0)

area_1 = integrate.trapezoid(signal[mask_1], time[mask_1])
area_2 = integrate.trapezoid(signal[mask_2], time[mask_2])

print(f"Area peak 1: {area_1:.3f} a.u.·min")
print(f"Area peak 2: {area_2:.3f} a.u.·min")
print(f"Area ratio peak 1 / peak 2: {area_1/area_2:.3f}")

```{admonition} Interpreting units
:class: note
An integral has the unit of $y$ multiplied by the unit of $x$. If the signal is measured in mAU and time in minutes, the peak area is measured in mAU·min. A calibration model can then relate the area to concentration or amount of substance.
```

## Further exploration: multiple integration

In some areas of chemistry, especially quantum chemistry and statistical thermodynamics, we encounter integrals over several variables. SciPy also provides functions such as `dblquad` and `tplquad`. It is useful to know that these exist, although they are not a main learning goal of this chapter.

In [ ]:
def g(y, x):
    return x*np.sin(y) - y*np.exp(x)

double_integral, error = integrate.dblquad(g, -1, 1, 0, np.pi/2)
print("Double integral:", double_integral)

## Short summary

- Numerical integration sums small contributions over an interval.
- Rectangle, trapezoidal and Simpson's methods use different approximations between points.
- A numerical result should be checked by changing the step size or the number of intervals.
- For experimental data, `trapezoid` and `simpson` are particularly useful.
- The unit and chemical meaning of the integral must always be interpreted together with the data.

## Exercises

```{admonition} Exercise 1 – rectangle methods
:class: tip
Integrate $f(x)=x^2-2x+4$ from 2 to 8 using the left, right and midpoint approximations. First use $n=10$ and then $n=100$. Compare with the analytical value.
```

```{admonition} Exercise 2 – convergence
:class: tip
Make a plot of absolute error as a function of $n$ for the left approximation, midpoint approximation and trapezoidal method. Use logarithmic axes.
```

```{admonition} Exercise 3 – chromatographic peak
:class: tip
Create a Gaussian-shaped peak centred at 5.0 min and add weak random noise. Integrate the peak with the trapezoidal method. How does the area change if you move the integration limits?
```

```{admonition} Exercise 4 – NMR
:class: tip
Two NMR signals have numerical areas 3.02 and 1.01. What might the area ratio suggest about the relative numbers of hydrogen atoms if the responses can be compared directly?
```

```{admonition} Exercise 5 – heat from power data
:class: tip
A calorimeter records heat power $P(t)$ in watts every second. Explain why the integral $\int P(t)\,dt$ gives energy, and write code that integrates a synthetic dataset. What unit does the answer have?
```

```{admonition} Exercise 6 – choosing a method
:class: tip
When would you use `quad`, and when would you use `trapezoid`? Give one chemical example of each.
```

```{admonition} Exercise 7 – Simpson
:class: tip
Implement Simpson's method yourself. Compare with `scipy.integrate.simpson` for $f(x)=\cos x + 2$ over the interval $[2,12]$.
```

```{admonition} Exercise 8 – a difficult function
:class: tip
Study $f(x)=\sin(1/x)$ close to $x=0$. Plot the function and investigate how different integration limits and resolutions affect the result. Explain why this is a numerically challenging problem.
```